# RQ1 locked post-hoc analysis

This notebook consumes the completed S1 result at `/kaggle/input/notebooks/dyhngg/test-rq1`. It performs no training and does not open any additional test checkpoints. The learned 128-D projection is the primary representation; backbone and fixed-random projection results are retained as robustness supplements.

## Secure repository checkout
Create a Kaggle secret named `github_token` containing a GitHub token with read access to the repository. `KAGGLE_API_TOKEN` is not needed because the source notebook output is attached directly. A GPU accelerator is not required.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from IPython.display import Image, Markdown, display
from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate>=0.9'], check=True)

## Locate and validate the completed S1 output

In [ ]:
INPUT_ROOT = Path('/kaggle/input/notebooks/dyhngg/test-rq1')
assert INPUT_ROOT.exists(), f'Attach the notebook output: {INPUT_ROOT}'
REQUIRED_SOURCE_FILES = (
    'specialization_table.csv',
    'representation_local_geometry_all_seeds.csv',
    'shared_dense_metrics_all_seeds.csv',
)
# Analysis only needs the finalized CSVs. Do not use the stricter resume-training
# materializer, which also requires every checkpoint and an exported ZIP.
SOURCE_CANDIDATES = sorted({
    path.parent
    for path in INPUT_ROOT.rglob('specialization_table.csv')
    if all((path.parent / filename).is_file() for filename in REQUIRED_SOURCE_FILES)
})
assert len(SOURCE_CANDIDATES) == 1, (
    f'Expected exactly one finalized S1 result containing {REQUIRED_SOURCE_FILES}; '
    f'found {SOURCE_CANDIDATES}. Available CSVs: '
    f'{[str(path.relative_to(INPUT_ROOT)) for path in INPUT_ROOT.rglob(chr(42) + chr(46) + "csv")][:50]}'
)
SOURCE_ROOT = SOURCE_CANDIDATES[0]
print('Validated source:', SOURCE_ROOT)

## Run the four locked analyses
LOSO scaling is fit inside each training fold. Bootstrap resamples complete seed blocks, keeping all four widths from a seed together.

In [ ]:
import importlib
import rq1_analysis
rq1_analysis = importlib.reload(rq1_analysis)

OUTPUT_DIR = Path('/kaggle/working/rq1-posthoc-analysis')
started = time.perf_counter()
result = rq1_analysis.run_rq1_analysis(SOURCE_ROOT, OUTPUT_DIR, bootstrap_replicates=5000)
print(f'Completed in {time.perf_counter() - started:.1f} seconds')
print(result['decision'])

## Primary numerical results

In [ ]:
import pandas as pd

matched = pd.read_csv(OUTPUT_DIR / 'rq1_matched_pair_040_060.csv')
loso = pd.read_csv(OUTPUT_DIR / 'rq1_loso_predictors.csv')
bootstrap = pd.read_csv(OUTPUT_DIR / 'rq1_bootstrap_correlations.csv')
display(Markdown('### Matched pair: learned projection'))
display(matched)
display(Markdown('### LOSO: learned projection'))
display(loso[loso.representation.eq('learned_projection')].sort_values('loso_mae'))
display(Markdown('### Seed-block bootstrap: learned projection'))
display(bootstrap[bootstrap.representation.eq('learned_projection')])
display(Markdown((OUTPUT_DIR / 'rq1_summary.md').read_text()))

## Figures

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'rq1_dense_geometry_vs_accuracy.png')))
display(Image(filename=str(OUTPUT_DIR / 'rq1_geometry_vs_resource_residual.png')))

## Validate and export

In [ ]:
REQUIRED = [
    'rq1_matched_pair_040_060.csv',
    'rq1_loso_predictors.csv',
    'rq1_resource_residuals.csv',
    'rq1_bootstrap_correlations.csv',
    'rq1_summary.md',
    'rq1_dense_geometry_vs_accuracy.png',
    'rq1_geometry_vs_resource_residual.png',
]
for filename in REQUIRED:
    path = OUTPUT_DIR / filename
    assert path.is_file() and path.stat().st_size > 0, f'Missing/empty output: {path}'
bundle_path = Path('/kaggle/working/rq1-posthoc-analysis.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
print('Download:', bundle_path)
print('Required outputs:', REQUIRED)